# Week 1 Task: Python for Machine Learning & Data Preprocessing

**Internship:** AI Pioneers Internship – Machine Learning  
**Task:** Python for Machine Learning & Data Preprocessing  
**Dataset:** Disease Prediction Health Dataset

## Objective
This notebook demonstrates loading, inspecting, cleaning, encoding, feature scaling, feature selection, exploratory data analysis (EDA), and exporting a processed dataset using Python, Pandas, NumPy, Matplotlib, Seaborn, and Scikit-learn.

> The original dataset contains 3,000 records and 10 columns. The target variable is `Disease`.


In [ ]:
# 1. Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif

pd.set_option("display.max_columns", None)
sns.set_theme()

print("Libraries imported successfully.")


In [ ]:
# 2. Load the dataset
DATA_PATH = "../data/raw/Disease_Prediction_Health_Dataset.xlsx"

df = pd.read_excel(DATA_PATH, sheet_name="Health_Data")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Basic dataset inspection
print("Data types:")
display(df.dtypes)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)


In [ ]:
# 4. Check missing values and duplicate rows
missing_values = df.isnull().sum().sort_values(ascending=False)
duplicate_count = df.duplicated().sum()

print("Missing values by column:")
display(missing_values)

print("Total duplicate rows:", duplicate_count)

if missing_values.sum() == 0:
    print("Result: No missing values are present in the current dataset.")
if duplicate_count == 0:
    print("Result: No duplicate rows are present in the current dataset.")


In [ ]:
# 5. Separate features and target
X = df.drop(columns=["Disease"])
y = df["Disease"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
display(y.value_counts())


In [ ]:
# 6. Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:
# 7. Exploratory Data Analysis (EDA): target distribution
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Disease")
plt.title("Disease Class Distribution")
plt.xlabel("Disease")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


In [ ]:
# 8. EDA: numerical feature distributions
df[numeric_features].hist(figsize=(14, 10), bins=20)
plt.suptitle("Numerical Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 9. EDA: correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_features].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:
# 10. EDA: categorical feature distributions
fig, axes = plt.subplots(1, len(categorical_features), figsize=(16, 5))

for ax, col in zip(axes, categorical_features):
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(f"{col} Distribution")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 11. Train/Test Split

The split is performed **before fitting preprocessing transformations** so that information from the test set does not influence the preprocessing fitted on the training set. This helps avoid data leakage.


In [ ]:
# Encode the target only; feature encoding is handled by the preprocessing pipeline.
y_encoded = (y == "Disease").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:
# 12. Build preprocessing pipeline
# Numerical: median imputation + StandardScaler
# Categorical: most-frequent imputation + OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Features after encoding/scaling:", len(feature_names))
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


In [ ]:
# 13. Feature selection using Mutual Information
# Select the 8 most informative transformed features.
k = min(8, X_train_processed.shape[1])

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=k
)

X_train_selected = selector.fit_transform(X_train_processed, y_train)
X_test_selected = selector.transform(X_test_processed)

selected_features = feature_names[selector.get_support()]

print("Selected features:")
for feature in selected_features:
    print("-", feature)

print("\nSelected training shape:", X_train_selected.shape)
print("Selected testing shape:", X_test_selected.shape)


In [ ]:
# 14. Verify scaling on numerical features
scaled_train_df = pd.DataFrame(
    X_train_processed[:, :len(numeric_features)],
    columns=numeric_features
)

print("Approximate means after StandardScaler:")
display(scaled_train_df.mean().round(3))

print("\nApproximate standard deviations after StandardScaler:")
display(scaled_train_df.std(ddof=0).round(3))


In [ ]:
# 15. Create the final processed dataset using the full dataset
# This exported artifact is for demonstrating the final preprocessing result.
# For model training/evaluation, always fit preprocessing only on the training set.

X_full_processed = preprocessor.fit_transform(X)
feature_names_full = preprocessor.get_feature_names_out()

selector_full = SelectKBest(
    score_func=mutual_info_classif,
    k=min(8, X_full_processed.shape[1])
)

X_full_selected = selector_full.fit_transform(X_full_processed, y_encoded)
selected_full_names = feature_names_full[selector_full.get_support()]

processed_df = pd.DataFrame(
    X_full_selected,
    columns=selected_full_names
)
processed_df["Disease"] = y_encoded

output_path = "../data/processed/cleaned_dataset.csv"
processed_df.to_csv(output_path, index=False)

print("Processed dataset saved to:", output_path)
display(processed_df.head())


In [ ]:
# 16. Final validation
final_df = pd.read_csv("../data/processed/cleaned_dataset.csv")

print("Final processed shape:", final_df.shape)
print("Missing values in processed dataset:", final_df.isnull().sum().sum())
print("Duplicate rows in processed dataset:", final_df.duplicated().sum())
display(final_df.head())


## Conclusion

The dataset was loaded with Pandas and inspected using standard descriptive and structural checks. Missing values and duplicates were checked. Exploratory analysis was performed for the target, numerical variables, and categorical variables. Numerical features were median-imputed and standardized using `StandardScaler`, while categorical features were imputed and one-hot encoded. Mutual-information-based `SelectKBest` was then used for feature selection. The resulting processed dataset was exported as a CSV file for downstream machine-learning work.

This workflow satisfies the Week 1 requirements for Python, NumPy, Pandas, data cleaning, missing-value handling, categorical encoding, normalization/scaling, feature selection, EDA, and documentation.
